## Project Objective

The objective of this project is to predict concrete compressive strength using information about the concrete mixture and curing age.

This is a supervised regression problem because the target variable, compressive strength, is continuous.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [3]:
pip install xlrd

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.


In [7]:
file_path = Path("../Raw_Data/Concrete_Data.xls")

df_concrete = pd.read_excel(file_path)

df_concrete.head()

,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [9]:
df_concrete.columns

Index(['Cement (component 1)(kg in a m^3 mixture)',
       'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
       'Fly Ash (component 3)(kg in a m^3 mixture)',
       'Water  (component 4)(kg in a m^3 mixture)',
       'Superplasticizer (component 5)(kg in a m^3 mixture)',
       'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
       'Fine Aggregate (component 7)(kg in a m^3 mixture)', 'Age (day)',
       'Concrete compressive strength(MPa, megapascals) '],
      dtype='object')

In [11]:
def normalize_text(text):
    if pd.isna(text):
        return text

    if not isinstance(text, str):
        return text

    return text.lower().strip().replace(" ", "_")


In [15]:
df_concrete.columns = [
    normalize_text(column)
    for column in df_concrete.columns
]

In [16]:
df_concrete.columns

Index(['cement_(component_1)(kg_in_a_m^3_mixture)',
       'blast_furnace_slag_(component_2)(kg_in_a_m^3_mixture)',
       'fly_ash_(component_3)(kg_in_a_m^3_mixture)',
       'water__(component_4)(kg_in_a_m^3_mixture)',
       'superplasticizer_(component_5)(kg_in_a_m^3_mixture)',
       'coarse_aggregate__(component_6)(kg_in_a_m^3_mixture)',
       'fine_aggregate_(component_7)(kg_in_a_m^3_mixture)', 'age_(day)',
       'concrete_compressive_strength(mpa,_megapascals)'],
      dtype='object')

In [17]:
df_concrete.dtypes


cement_(component_1)(kg_in_a_m^3_mixture)                float64
blast_furnace_slag_(component_2)(kg_in_a_m^3_mixture)    float64
fly_ash_(component_3)(kg_in_a_m^3_mixture)               float64
water__(component_4)(kg_in_a_m^3_mixture)                float64
superplasticizer_(component_5)(kg_in_a_m^3_mixture)      float64
coarse_aggregate__(component_6)(kg_in_a_m^3_mixture)     float64
fine_aggregate_(component_7)(kg_in_a_m^3_mixture)        float64
age_(day)                                                  int64
concrete_compressive_strength(mpa,_megapascals)          float64
dtype: object

In [18]:
df_concrete.head()

,cement_(component_1)(kg_in_a_m^3_mixture),blast_furnace_slag_(component_2)(kg_in_a_m^3_mixture),fly_ash_(component_3)(kg_in_a_m^3_mixture),water__(component_4)(kg_in_a_m^3_mixture),superplasticizer_(component_5)(kg_in_a_m^3_mixture),coarse_aggregate__(component_6)(kg_in_a_m^3_mixture),fine_aggregate_(component_7)(kg_in_a_m^3_mixture),age_(day),"concrete_compressive_strength(mpa,_megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [19]:
df_concrete.isnull().sum()

cement_(component_1)(kg_in_a_m^3_mixture)                0
blast_furnace_slag_(component_2)(kg_in_a_m^3_mixture)    0
fly_ash_(component_3)(kg_in_a_m^3_mixture)               0
water__(component_4)(kg_in_a_m^3_mixture)                0
superplasticizer_(component_5)(kg_in_a_m^3_mixture)      0
coarse_aggregate__(component_6)(kg_in_a_m^3_mixture)     0
fine_aggregate_(component_7)(kg_in_a_m^3_mixture)        0
age_(day)                                                0
concrete_compressive_strength(mpa,_megapascals)          0
dtype: int64

In [22]:
df_concrete.rename(
    columns={
        "blast_furnace_slag_(component_2)(kg_in_a_m^3_mixture)": "blast_furnace_slag"
    },
    inplace=True
)

In [24]:
zero_percentage = (
    df_concrete["blast_furnace_slag"].eq(0).mean() * 100
)

print(f"Zero values: {zero_percentage}")
print(f"Percentage zero: {zero_percentage:.1f}%")

Zero values: 45.24271844660194
Percentage zero: 45.2%


In [27]:
missing_summary = pd.DataFrame({
    "missing_count": df_concrete.isna().sum(),
    "missing_percentage": df_concrete.isna().mean() * 100
})

missing_summary

,missing_count,missing_percentage
cement_(component_1)(kg_in_a_m^3_mixture),0,0.0
blast_furnace_slag,0,0.0
fly_ash_(component_3)(kg_in_a_m^3_mixture),0,0.0
water__(component_4)(kg_in_a_m^3_mixture),0,0.0
superplasticizer_(component_5)(kg_in_a_m^3_mixture),0,0.0
coarse_aggregate__(component_6)(kg_in_a_m^3_mixture),0,0.0
fine_aggregate_(component_7)(kg_in_a_m^3_mixture),0,0.0
age_(day),0,0.0
"concrete_compressive_strength(mpa,_megapascals)",0,0.0


In [28]:
duplicate_count = df_concrete.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 25


In [29]:
df_concrete[df_concrete.duplicated(keep=False)].sort_values(
    by=list(df_concrete.columns)
)

,cement_(component_1)(kg_in_a_m^3_mixture),blast_furnace_slag,fly_ash_(component_3)(kg_in_a_m^3_mixture),water__(component_4)(kg_in_a_m^3_mixture),superplasticizer_(component_5)(kg_in_a_m^3_mixture),coarse_aggregate__(component_6)(kg_in_a_m^3_mixture),fine_aggregate_(component_7)(kg_in_a_m^3_mixture),age_(day),"concrete_compressive_strength(mpa,_megapascals)"
801,252.0,0.0,0.0,185.0,0.0,1111.0,784.0,28,19.691435
809,252.0,0.0,0.0,185.0,0.0,1111.0,784.0,28,19.691435
83,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
86,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
88,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
91,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
106,362.6,189.0,0.0,164.9,11.6,944.7,755.8,7,55.895819
109,362.6,189.0,0.0,164.9,11.6,944.7,755.8,7,55.895819
111,362.6,189.0,0.0,164.9,11.6,944.7,755.8,7,55.895819
129,362.6,189.0,0.0,164.9,11.6,944.7,755.8,28,71.298713
